In [ ]:
from music21 import converter

def analyze_midi(file):
    midi = converter.parse(file)

    notes = list(midi.flatten().notes)

    duration = midi.duration.quarterLength
    note_count = len(notes)

    density = note_count / duration if duration > 0 else 0

    tempos = midi.metronomeMarkBoundaries()
    bpm = tempos[0][2].number if tempos else "Unknown"

    print("Tempo (BPM):", bpm)
    print("Total Notes:", note_count)
    print("Duration:", duration)
    print("Note Density:", round(density, 2))


In [ ]:
analyze_midi("generated_Classical.mid")

In [ ]:
from music21 import converter
import numpy as np


In [ ]:
def extract_features(midi_file):
    midi = converter.parse(midi_file)
    notes = list(midi.flatten().notes)

    # --- Tempo ---
    tempos = midi.metronomeMarkBoundaries()
    bpm = tempos[0][2].number if tempos else 100

    # --- Duration ---
    duration = midi.duration.quarterLength

    # --- Note Density ---
    density = len(notes) / duration if duration > 0 else 0

    # --- Pitch Information ---
    pitches = []
    for n in notes:
        if n.isNote:
            pitches.append(n.pitch.midi)
        elif n.isChord:
            pitches.extend(p.midi for p in n.pitches)

    pitch_range = max(pitches) - min(pitches) if pitches else 0
    pitch_std = np.std(pitches) if pitches else 0

    return {
        "bpm": bpm,
        "density": density,
        "pitch_range": pitch_range,
        "pitch_std": pitch_std
    }


In [ ]:
GENRE_RULES = {
    "Classical": {"bpm": (60, 100), "density": (2, 6), "pitch_std": (10, 25)},
    "Jazz":      {"bpm": (90, 130), "density": (4, 9), "pitch_std": (15, 30)},
    "Rock":      {"bpm": (110, 160), "density": (6, 14), "pitch_std": (8, 20)},
    "EDM":       {"bpm": (120, 135), "density": (8, 16), "pitch_std": (5, 15)},
    "Hip-Hop":   {"bpm": (70, 100), "density": (4, 8), "pitch_std": (6, 18)},
    "Ambient":   {"bpm": (40, 80), "density": (1, 5), "pitch_std": (12, 28)}
}


In [ ]:
def calculate_accuracy(features, genre):
    rules = GENRE_RULES[genre]

    score = 0
    total_checks = 3

    def match(value, range_):
        return range_[0] <= value <= range_[1]

    if match(features["bpm"], rules["bpm"]):
        score += 1

    if match(features["density"], rules["density"]):
        score += 1

    if match(features["pitch_std"], rules["pitch_std"]):
        score += 1

    accuracy = (score / total_checks) * 100
    return accuracy


In [ ]:
file = "generated_Rock.mid"
expected_genre = "Rock"

features = extract_features(file)

print("Extracted Features:", features)

accuracy = calculate_accuracy(features, expected_genre)
print(f"Genre Accuracy: {accuracy:.2f}/100")


In [ ]:
import numpy as np
import pickle
from tensorflow.keras.models import load_model

# --- Load saved data ---
X_midi = np.load("../models/X_midi.npy")

with open("../models/int_to_note.pkl", "rb") as f:
    int_to_note = pickle.load(f)

midi_model = load_model("../models/midi_lstm_model.keras")


In [ ]:
def get_seed_by_genre(genre):
    if genre == "Classical":
        start = 0
    elif genre == "Jazz":
        start = len(X_midi)//6
    elif genre == "Rock":
        start = 2*len(X_midi)//6
    elif genre == "EDM":
        start = 3*len(X_midi)//6
    elif genre == "Hip-Hop":
        start = 4*len(X_midi)//6
    else:  # Ambient
        start = 5*len(X_midi)//6

    idx = np.random.randint(start, start + len(X_midi)//6)
    pattern = X_midi[idx]

    return pattern.reshape(1, len(pattern), 1)


In [ ]:
scores = []
genre = "Rock"

for i in range(5):
    seed = get_seed_by_genre(genre)
    notes = generate_music(seed, length=400)

    filename = f"eval_{i}.mid"
    save_midi(notes, genre, filename)

    features = extract_features(filename)
    acc = calculate_accuracy(features, genre)

    print(f"Run {i+1} Accuracy:", acc)
    scores.append(acc)

final_score = np.mean(scores)
variation = np.std(scores)

print("\nFinal Accuracy:", round(final_score,2), "/100")
print("Variation (±):", round(variation,2))
